In [1]:
from datasets import load_dataset

# Load the CoDET-M4 dataset
dataset = load_dataset("DaniilOr/CoDET-M4")

/home/shenghua/anaconda3/envs/cpg/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
"""
test_cpg.py
===========
External smoke test for generate_cpg.py
Tests one snippet per language (python / cpp / java) without modifying the original code.

Usage:
    python test_cpg.py
    python test_cpg.py --joern_path /home/shenghua/bin/joern-cli/
    python test_cpg.py --temp_dir ./my_temp
"""

import argparse
import shutil
from utils.Joern import JoernRunner, small_sample

# ── One minimal snippet per language ────────────────────────────────────────

SNIPPETS = {
    'python': """\
def add(a, b):
    return a + b

print(add(1, 2))
""",
    'cpp': """\
#include <iostream>
int add(int a, int b) { return a + b; }
int main() {
    std::cout << add(1, 2) << std::endl;
    return 0;
}
""",
    'java': """\
public class Add {
    public static int add(int a, int b) { return a + b; }
    public static void main(String[] args) {
        System.out.println(add(1, 2));
    }
}
""",
}

# ── Test runner ──────────────────────────────────────────────────────────────

def run_tests(joern_path=None, temp_dir="./temp_smoke_test"):
    runner  = JoernRunner(temp_dir=temp_dir, joern_path=joern_path)
    all_ok  = True

    print("\n" + "=" * 60)
    print("SMOKE TEST — one snippet per language")
    print("=" * 60)

    for lang, snippet in SNIPPETS.items():
        result = runner.parse_one(idx=0, code_string=snippet, language=lang)
        ok     = result['graphml'] is not None

        if ok:
            n_nodes = result['graphml'].count('<node ')
            n_edges = result['graphml'].count('<edge ')
            print(f"  [OK ✓]  {lang:<8}  nodes={n_nodes:>4}  edges={n_edges:>4}")
        else:
            print(f"  [FAIL ✗]  {lang:<8}  error: {result.get('error', 'unknown')}")
            all_ok = False

    print("=" * 60)
    print("Result:", "ALL PASSED ✓" if all_ok else "SOME FAILED ✗")
    print("=" * 60 + "\n")

    # Cleanup temp dir
    import os
    if os.path.exists(temp_dir):
        shutil.rmtree(temp_dir, ignore_errors=True)

    return all_ok


if __name__ == "__main__":
    joern_path = '/home/shenghua/bin/joern-cli/'
    temp_dir   = "./temp_smoke_test"
    ok = run_tests(joern_path=joern_path, temp_dir=temp_dir)
    exit(0 if ok else 1)


SMOKE TEST — one snippet per language
  [OK ✓]  python    nodes=  66  edges= 274
  [FAIL ✗]  cpp       error: Command '['/home/shenghua/bin/joern-cli/joern-parse', '/home/shenghua/CodeGraph/temp_smoke_test/sample_0.cpp', '-o', '/home/shenghua/CodeGraph/temp_smoke_test/sample_0.bin']' returned non-zero exit status 1.
  [OK ✓]  java      nodes=  75  edges= 269
Result: SOME FAILED ✗



In [2]:
python_dataset = dataset['train'].filter(lambda x: x['language'] == 'python')
java_dataset = dataset['train'].filter(lambda x: x['language'] == 'java')
# cpp_dataset = dataset['train'].filter(lambda x: x['language'] == 'cpp')
# train_dataset = dataset['train'].filter(lambda x: x['split'] == 'train')
# val_dataset = dataset['train'].filter(lambda x: x['split'] == 'val')
# test_dataset = dataset['train'].filter(lambda x: x['split'] == 'test')

In [12]:
java_dataset.filter(lambda x: x['split'] == 'train')

Filter: 100%|██████████| 174169/174169 [00:06<00:00, 26034.71 examples/s]


Dataset({
    features: ['code', 'language', 'model', 'split', 'target', 'source', 'features', 'cleaned_code', '__index_level_0__'],
    num_rows: 140875
})

py
train 149763
val 17685
test 17715

java
train 140875
val 16651
test 16643

In [6]:
java_dataset

Dataset({
    features: ['code', 'language', 'model', 'split', 'target', 'source', 'features', 'cleaned_code', '__index_level_0__'],
    num_rows: 174169
})

In [2]:
test_dataset 

Dataset({
    features: ['code', 'language', 'model', 'split', 'target', 'source', 'features', 'cleaned_code', '__index_level_0__'],
    num_rows: 47749
})

In [2]:
python_dataset

Dataset({
    features: ['code', 'language', 'model', 'split', 'target', 'source', 'features', 'cleaned_code', '__index_level_0__'],
    num_rows: 185163
})

In [4]:
import json

with open('./CPGs/raw/cpg_dataset.jsonl', 'r') as f:
    line_count = sum(1 for line in f)
print(f"Total lines: {line_count}")


Total lines: 5382


In [ ]:
with open('./CPGs/raw/cpg_dataset.jsonl', 'r') as f:
    for i, line in enumerate(f):
        if i == 1:
            break
        record = json.loads(line)
print(record['code'])
print(record['graphml'])

In [3]:
record

{'idx': 6,
 'target': 'human',
 'model': 'human',
 'language': 'cpp',
 'code': 'void insert_vec(int t, int x) {\n  for (int i = LOGN - 1; i >= 0; i--) basis[t][i] = basis[t - 1][i];\n  for (int i = LOGN - 1; i >= 0; i--) {\n\n if ((x >> i) & 1) {\n\n\nif (!basis[t][i]) {\n\n\n  basis[t][i] = x;\n\n\n  return;\n\n\n} else {\n\n\n  x ^= basis[t][i];\n\n\n}\n\n }\n  }\n}',
 'graphml': '<graphml \nxsi:schemaLocation="http://graphml.graphdrawing.org/xmlns http://graphml.graphdrawing.org/xmlns/1.0/graphml.xsd" xmlns="http://graphml.graphdrawing.org/xmlns" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <key id="labelV" for="node" attr.name="labelV" attr.type="string"/>\n  <key id="labelE" for="edge" attr.name="labelE" attr.type="string"/>\n  <key id="node__TYPE_DECL__AST_PARENT_FULL_NAME" for="node" attr.name="AST_PARENT_FULL_NAME" attr.type="string"/>\n  <key id="node__TYPE_DECL__COLUMN_NUMBER" for="node" attr.name="COLUMN_NUMBER" attr.type="int"/>\n  <key id="node__NAMESPACE_BLOC

In [ ]:
from utils.cpg2homo import CPGHomoDataset
dataset = CPGHomoDataset(
    root='./CPGs',
    force_reload=False
)

Processing...


Processing file: CPGJ/raw/cpg_dataset.jsonl
Parsing 10 graphs with FULL attributes...


100%|██████████| 10/10 [00:00<00:00, 148.03it/s]

Successfully loaded 10 graphs.
Saved to CPGJ/processed_homo/processed.pt



Done!
/home/shenghua/CodeGraph/utils/cpg2homo_org.py:119: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.load(self.processed_paths[0])


In [12]:
dataset.data

/tmp/ipykernel_164707/802427235.py:1: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  dataset.data


Data(edge_index=[2, 1280], num_nodes=352, node_texts=[10], edge_texts=[10], target=[10], model=[10], code=[10], dataset_idx=[10])

In [1]:
import torch
from utils.cpg2hetero import CPGHeteroDataset
dataset = CPGHeteroDataset(
    root='./CPGh',
    force_reload=False)
dataset

/home/shenghua/anaconda3/envs/cpg/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/shenghua/CodeGraph/utils/cpg2hetero.py:193: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you 

CPGHeteroDataset(33342)

In [1]:
import torch
from utils.cpg2hetero import CPGHeteroDataset
dataset = CPGHeteroDataset(
    root='./CPGs',
    force_reload=False)
dataset

/home/shenghua/anaconda3/envs/cpg/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/shenghua/CodeGraph/utils/cpg2hetero.py:193: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you 

CPGHeteroDataset(77332)

In [2]:
dataset.data

/tmp/ipykernel_1649901/802427235.py:1: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  dataset.data


HeteroData(
  target=[77332],
  model=[77332],
  language=[77332],
  source=[77332],
  hash=[77332],
  dataset_idx=[77332],
  code=[77332],
  split=[77332],
  FILE={ x=[136511, 768] },
  META_DATA={ x=[77332, 768] },
  NAMESPACE={ x=[77676, 768] },
  NAMESPACE_BLOCK={ x=[174971, 768] },
  TYPE={ x=[408679, 768] },
  TYPE_DECL={ x=[409811, 768] },
  METHOD_PARAMETER_OUT={ x=[1483727, 768] },
  IMPORT={ x=[54546, 768] },
  METHOD_PARAMETER_IN={ x=[2029518, 768] },
  LITERAL={ x=[884448, 768] },
  METHOD_RETURN={ x=[1087866, 768] },
  CLOSURE_BINDING={ x=[140141, 768] },
  ANNOTATION_LITERAL={ x=[317, 768] },
  LOCAL={ x=[948905, 768] },
  MODIFIER={ x=[431212, 768] },
  ANNOTATION_PARAMETER={ x=[335, 768] },
  METHOD_REF={ x=[61992, 768] },
  TYPE_REF={ x=[172973, 768] },
  CONTROL_STRUCTURE={ x=[331908, 768] },
  FIELD_IDENTIFIER={ x=[549517, 768] },
  METHOD={ x=[1087866, 768] },
  CALL={ x=[3639519, 768] },
  ANNOTATION_PARAMETER_ASSIGN={ x=[335, 768] },
  MEMBER={ x=[265797, 768] },


In [2]:
dataset.data

/tmp/ipykernel_1649725/802427235.py:1: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  dataset.data


HeteroData(
  target=[33342],
  model=[33342],
  language=[33342],
  source=[33342],
  hash=[33342],
  dataset_idx=[33342],
  code=[33342],
  split=[33342],
  FILE={ x=[56758, 768] },
  META_DATA={ x=[33342, 768] },
  NAMESPACE={ x=[33448, 768] },
  NAMESPACE_BLOCK={ x=[73467, 768] },
  TYPE={ x=[160135, 768] },
  TYPE_DECL={ x=[160880, 768] },
  JUMP_TARGET={ x=[156, 768] },
  METHOD_PARAMETER_OUT={ x=[572999, 768] },
  MODIFIER={ x=[157857, 768] },
  CLOSURE_BINDING={ x=[72984, 768] },
  METHOD_REF={ x=[30782, 768] },
  CALL={ x=[1541371, 768] },
  FIELD_IDENTIFIER={ x=[254859, 768] },
  ANNOTATION_LITERAL={ x=[131, 768] },
  TAG={ x=[13640, 768] },
  IDENTIFIER={ x=[1531661, 768] },
  ANNOTATION={ x=[578, 768] },
  TYPE_REF={ x=[71052, 768] },
  ARRAY_INITIALIZER={ x=[2, 768] },
  ANNOTATION_PARAMETER={ x=[133, 768] },
  METHOD_RETURN={ x=[443095, 768] },
  METHOD_PARAMETER_IN={ x=[843104, 768] },
  BLOCK={ x=[676467, 768] },
  LOCAL={ x=[436622, 768] },
  METHOD={ x=[443095, 768] }

In [5]:
train_list = dataset.get_subset(language='python')

/home/shenghua/CodeGraph/utils/cpg2hetero.py:359: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. The data of the dataset is already cached, so any modifications to `data` will not be reflected when accessing its elements. Clearing the cache now by removing all elements in `dataset._data_list`. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  num_graphs = len(self.data.dataset_idx)
/home/shenghua/CodeGraph/utils/cpg2hetero.py:363: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual a

In [4]:
from torch_geometric.loader import DataLoader
train_loader = DataLoader(train_list, batch_size=32, shuffle=True)

In [5]:
for batch in train_loader:
    print(batch)
    break

HeteroDataBatch(
  target=[32],
  model=[32],
  language=[32],
  source=[32],
  hash=[32],
  dataset_idx=[32],
  code=[32],
  split=[32],
  BINDING={
    x=[86, 768],
    batch=[86],
    ptr=[33],
  },
  BLOCK={
    x=[948, 768],
    batch=[948],
    ptr=[33],
  },
  CALL={
    x=[2116, 768],
    batch=[2116],
    ptr=[33],
  },
  FIELD_IDENTIFIER={
    x=[394, 768],
    batch=[394],
    ptr=[33],
  },
  FILE={
    x=[64, 768],
    batch=[64],
    ptr=[33],
  },
  IDENTIFIER={
    x=[2203, 768],
    batch=[2203],
    ptr=[33],
  },
  LITERAL={
    x=[402, 768],
    batch=[402],
    ptr=[33],
  },
  LOCAL={
    x=[790, 768],
    batch=[790],
    ptr=[33],
  },
  META_DATA={
    x=[32, 768],
    batch=[32],
    ptr=[33],
  },
  METHOD={
    x=[548, 768],
    batch=[548],
    ptr=[33],
  },
  METHOD_PARAMETER_IN={
    x=[1063, 768],
    batch=[1063],
    ptr=[33],
  },
  METHOD_PARAMETER_OUT={
    x=[502, 768],
    batch=[502],
    ptr=[33],
  },
  METHOD_RETURN={
    x=[548, 768],
    ba

In [6]:
classes = set(dataset.data.model)
num_classes = len(classes)
label_map = {label: idx for idx, label in enumerate(sorted(classes))}

/tmp/ipykernel_1385773/3434649277.py:1: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. The data of the dataset is already cached, so any modifications to `data` will not be reflected when accessing its elements. Clearing the cache now by removing all elements in `dataset._data_list`. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  classes = set(dataset.data.model)


In [9]:
label_map

{'codellama': 0,
 'gpt': 1,
 'human': 2,
 'llama3.1': 3,
 'nxcode': 4,
 'qwen1.5': 5}

In [7]:
classes 

{'codellama', 'gpt', 'human', 'llama3.1', 'nxcode', 'qwen1.5'}

In [8]:
torch.tensor([label_map[label] for label in batch.model])

tensor([3, 2, 2, 3, 1, 1, 0, 4, 2, 5, 4, 0, 1, 4, 2, 4, 1, 0, 5, 5, 0, 1, 2, 2,
        2, 4, 5, 2, 0, 2, 5, 3])